In [1]:
import os

# Ensure we're always running from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())
print(os.listdir("."))

/groups/nils/members/andras/scrna_project
['workflow', '.git', '.gitattributes', 'README.md', 'logs', 'results', 'config', '.snakemake', 'environment.yml', '.vscode', 'run_pipeline.py', 'notebooks', 'data', '.gitignore']


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set these two values to match the run you want to explore.
# RUN_ID must match the run_id used when the pipeline was executed.
# SAMPLE  must match the sample ID (FASTQ filename prefix).

RUN_ID = "SRR5071657_hg38"
SAMPLE = "SRR5071657"

# Derived paths — no need to change these
BAM     = f"results/{RUN_ID}/bam/{SAMPLE}.sorted.bam"
RAW_VCF = f"results/{RUN_ID}/vcf/{SAMPLE}.raw.vcf"
FLT_VCF = f"results/{RUN_ID}/vcf/{SAMPLE}.filtered.vcf"

print(f"Run:    {RUN_ID}")
print(f"Sample: {SAMPLE}")
print(f"BAM:    {BAM}  (exists: {os.path.exists(BAM)})")
print(f"VCF:    {FLT_VCF}  (exists: {os.path.exists(FLT_VCF)})")

Run:    SRR5071657_hg38
Sample: SRR5071657
BAM:    results/SRR5071657_hg38/bam/SRR5071657.sorted.bam  (exists: True)
VCF:    results/SRR5071657_hg38/vcf/SRR5071657.filtered.vcf  (exists: True)


In [3]:
import subprocess

# 1. How many reads actually aligned?
result = subprocess.run(
    ["samtools", "flagstat", BAM],
    capture_output=True, text=True
)
print(result.stdout)

37714055 + 0 in total (QC-passed reads + QC-failed reads)
34067020 + 0 primary
3647035 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
37714055 + 0 mapped (100.00% : N/A)
34067020 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)



In [4]:
# 2. Peek at the raw VCF — header + first variant
with open(RAW_VCF) as f:
    for line in f:
        print(line.strip())
        if not line.startswith("#"):
            break

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##bcftoolsVersion=1.23.1+htslib-1.23.1
##bcftoolsCommand=mpileup -f data/reference/genome.fa -q 20 -Q 20 --output-type b -o results/SRR5071657_hg38/vcf/SRR5071657.mpileup.bcf results/SRR5071657_hg38/bam/SRR5071657.sorted.bam
##reference=file://data/reference/genome.fa
##contig=<ID=chr1,length=248956422>
##contig=<ID=chr10,length=133797422>
##contig=<ID=chr11,length=135086622>
##contig=<ID=chr12,length=133275309>
##contig=<ID=chr13,length=114364328>
##contig=<ID=chr14,length=107043718>
##contig=<ID=chr15,length=101991189>
##contig=<ID=chr16,length=90338345>
##contig=<ID=chr17,length=83257441>
##contig=<ID=chr18,length=80373285>
##contig=<ID=chr19,length=58617616>
##contig=<ID=chr2,length=242193529>
##contig=<ID=chr20,length=64444167>
##contig=<ID=chr21,length=46709983>
##contig=<ID=chr22,length=50818468>
##contig=<ID=chr3,length=198295559>
##contig=<ID=chr4,length=190214555>
##contig=<ID=chr5,length=181538259>
##co

In [5]:
# 3. Count variants at each stage
def count_variants(vcf_path):
    result = subprocess.run(
        f"grep -v '^#' {vcf_path} | wc -l",
        shell=True, capture_output=True, text=True
    )
    return int(result.stdout.strip())

raw      = count_variants(RAW_VCF)
filtered = count_variants(FLT_VCF)

print(f"Raw variants:      {raw}")
print(f"Filtered variants: {filtered}")
print(f"Removed by filter: {raw - filtered}")

Raw variants:      53894
Filtered variants: 35294
Removed by filter: 18600


In [6]:
# 4. Look at the actual variant lines
with open(FLT_VCF) as f:
    for line in f:
        if not line.startswith("#"):
            fields = line.strip().split("\t")
            print(f"Chr: {fields[0]}  Pos: {fields[1]}  "
                  f"Ref: {fields[3]}  Alt: {fields[4]}  "
                  f"Qual: {fields[5]}  Filter: {fields[6]}")

Chr: chr1  Pos: 187485  Ref: G  Alt: A  Qual: 48.6479  Filter: PASS
Chr: chr1  Pos: 348250  Ref: G  Alt: T  Qual: 38.415  Filter: PASS
Chr: chr1  Pos: 348251  Ref: G  Alt: C  Qual: 38.415  Filter: PASS
Chr: chr1  Pos: 348252  Ref: A  Alt: T  Qual: 38.415  Filter: PASS
Chr: chr1  Pos: 348255  Ref: T  Alt: C  Qual: 38.415  Filter: PASS
Chr: chr1  Pos: 629273  Ref: T  Alt: C  Qual: 24.1272  Filter: PASS
Chr: chr1  Pos: 629276  Ref: C  Alt: T  Qual: 22.1507  Filter: PASS
Chr: chr1  Pos: 631862  Ref: G  Alt: A  Qual: 179.416  Filter: PASS
Chr: chr1  Pos: 633083  Ref: A  Alt: G  Qual: 26.4242  Filter: PASS
Chr: chr1  Pos: 633148  Ref: G  Alt: C  Qual: 119.209  Filter: PASS
Chr: chr1  Pos: 727717  Ref: G  Alt: C  Qual: 39.4149  Filter: PASS
Chr: chr1  Pos: 827252  Ref: T  Alt: A  Qual: 30.4183  Filter: PASS
Chr: chr1  Pos: 841742  Ref: A  Alt: T  Qual: 27.4222  Filter: PASS
Chr: chr1  Pos: 852019  Ref: G  Alt: T  Qual: 19.4636  Filter: PASS
Chr: chr1  Pos: 857100  Ref: C  Alt: T  Qual: 19.463

In [7]:
# 5. How many reads aligned at all?
result = subprocess.run(["samtools", "flagstat", BAM], capture_output=True, text=True)
print(result.stdout)

37714055 + 0 in total (QC-passed reads + QC-failed reads)
34067020 + 0 primary
3647035 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
37714055 + 0 mapped (100.00% : N/A)
34067020 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)



In [8]:
import subprocess

# 6. How deep is the coverage at any position?
result = subprocess.run(
    ["samtools", "coverage", BAM],
    capture_output=True, text=True
)
print(result.stdout)

#rname	startpos	endpos	numreads	covbases	coverage	meandepth	meanbaseq	meanmapq
chr1	1	248956422	2870456	11588678	4.6549	1.14522	35.9	222
chr10	1	133797422	1267543	5930971	4.4328	0.941032	35.9	245
chr11	1	135086622	2725411	7808078	5.78005	2.0031	35.9	249
chr12	1	133275309	2718547	7389088	5.54423	2.02631	35.9	235
chr13	1	114364328	404931	2509159	2.194	0.351642	35.9	239
chr14	1	107043718	744900	3828847	3.5769	0.691115	35.9	247
chr15	1	101991189	744717	3919996	3.84347	0.7251	35.9	244
chr16	1	90338345	1308570	5687847	6.29616	1.43887	35.9	237
chr17	1	83257441	1935122	7144451	8.58116	2.3087	35.9	245
chr18	1	80373285	284259	2068941	2.57417	0.351324	35.9	245
chr19	1	58617616	1924470	6651518	11.3473	3.25921	35.9	250
chr2	1	242193529	2483477	10155351	4.19307	1.0185	35.9	247
chr20	1	64444167	1156708	3905718	6.06062	1.78267	35.9	252
chr21	1	46709983	295821	1444966	3.09348	0.629156	35.9	169
chr22	1	50818468	737938	3250793	6.39687	1.44229	35.9	246
chr3	1	198295559	1349587	7606607	3.83599	0.676019	35.

In [9]:
# 7. Does the BAM actually have reads in it?
result = subprocess.run(["samtools", "view", "-c", BAM], capture_output=True, text=True)
print(result.stdout)

37714055

